# 🍳 Agent Tools & Thread-Scoped Conversational Memory

### Overview & Learning Objectives
In modern agentic architectures, an LLM shifts from a passive completion engine to an **active decision-maker** that plans, calls external tools, and maintains state across conversational turns.

This lab explores two fundamental building blocks of autonomous AI systems:
1. **Custom Tool Definition (`@tool`)**: Equipping the model with real-time web search capabilities via Tavily.
2. **Thread-Scoped State Persistence (`InMemorySaver`)**: Managing conversational memory so that the agent remembers previous turns within an isolated session identified by a `thread_id`.
3. **Groq Acceleration**: Powering the agent with Groq's high-speed **`llama-3.3-70b-versatile`** model for rapid reasoning and tool execution.

## 📐 System Architecture

The diagram below illustrates the end-to-end flow: the user query triggers the agent, which queries the checkpointer for session history, decides whether to invoke external search tools, and updates memory upon completion.

<div align="center">
  <img src="images/01_agent_tools_memory.png" alt="Agent Tools & Thread-Scoped Memory Architecture" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
flowchart TD
    User([👤 User Request]) --> Agent[🤖 Chef Agent<br/>Groq llama-3.3-70b-versatile]
    Agent <-->|1. Load / Save Checkpoint| Memory[(💾 InMemorySaver<br/>thread_id: '1')]
    Agent -->|2. Evaluate Tool Need| Decision{Requires Web Info?}
    Decision -->|Yes: Search Web| ToolNode[🛠️ web_search Tool]
    ToolNode <-->|API Request / Response| Tavily[(🌐 Tavily Search Engine)]
    ToolNode -->|Tool Results ToolMessage| Agent
    Decision -->|No: Synthesis Ready| FinalMsg[💬 AIMessage Response]
    FinalMsg --> User
```

</details>


## 1. Environment & API Configuration

We initialize environment variables using `python-dotenv`. For this lab, you need:
- `GROQ_API_KEY`: To access Groq's low-latency inference endpoint.
- `TAVILY_API_KEY`: To perform web searches via Tavily's AI search API.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify that required keys are present
groq_key = os.getenv("GROQ_API_KEY")
tavily_key = os.getenv("TAVILY_API_KEY")

if not groq_key:
    print("⚠️ Warning: GROQ_API_KEY not found in environment. Set it before invoking Groq.")
else:
    print("✅ GROQ_API_KEY loaded successfully.")

if not tavily_key:
    print("⚠️ Warning: TAVILY_API_KEY not found in environment. Web search tool requires this key.")
else:
    print("✅ TAVILY_API_KEY loaded successfully.")

## 2. Defining Custom Tools with `@tool`

LangChain's `@tool` decorator transforms a regular Python function into an agent-callable tool. Under the hood, it inspects:
- **Function Name**: Identified by the LLM when deciding which tool to call.
- **Type Annotations (`query: str -> Dict[str, Any]`)**: Form the JSON schema for parameter validation.
- **Docstring (`"""Search the web..."""`)**: Serves as the instructions for the LLM explaining **what** the tool does and **when** it should be chosen.

In [ ]:
from typing import Dict, Any
from langchain.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for up-to-date information, recipes, and ingredients."""
    return tavily_client.search(query)

## 3. Agent System Prompt Specification

The system prompt guides the LLM's persona, boundary constraints, and decision-making protocol. Here we establish a personal chef assistant focused on repurposing leftover household ingredients.

In [ ]:
system_prompt = """
You are a personal chef. The user will give you a list of ingredients they have left over in their house.

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.
"""

## 4. Agent Initialization with Groq & `InMemorySaver`

We construct the agent using `create_agent` from LangChain, configured with:
- **LLM**: `ChatGroq(model="llama-3.3-70b-versatile")` for ultra-fast, high-quality reasoning.
- **Tools**: `[web_search]`.
- **Checkpointer**: `InMemorySaver()`. This checkpointer snapshots the agent's internal state graph at every superstep, indexed by a unique `thread_id`.

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Initialize Groq Chat Model
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

# Compile agent with tools and in-memory checkpointer
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

## 5. Execution Flow & Memory Lifecycle

The sequence diagram below details the conversational lifecycle across turns for a given thread:

<div align="center">
  <img src="images/seq_agent_memory.png" alt="Conversational Memory Lifecycle Sequence Diagram" width="100%" />
</div>

<br/>

<details>
<summary><b>🔍 View Raw Mermaid Diagram Syntax</b></summary>

```mermaid
sequenceDiagram
    autonumber
    actor User
    participant Agent as Chef Agent (ChatGroq)
    participant Tool as web_search Tool
    participant Memory as InMemorySaver (thread_id: 1)

    User->>Agent: Turn 1: 'Leftover chicken and rice...'
    Agent->>Memory: Load checkpoints for thread_id '1'
    Memory-->>Agent: Initial Empty State
    Agent->>Agent: Analyze ingredients & formulate search query
    Agent->>Tool: web_search('leftover chicken and rice recipes')
    Tool-->>Agent: Return candidate recipe search results
    Agent->>Memory: Persist state (HumanMessage, AIMessage, ToolMessage)
    Agent-->>User: Suggests recipes (e.g. Chicken Fried Rice)

    Note over User,Memory: Subsequent turn retains memory context via thread_id
    User->>Agent: Turn 2: 'Give me instructions for the first one'
    Agent->>Memory: Load checkpoints for thread_id '1'
    Memory-->>Agent: Full history of Turn 1 returned
    Agent-->>User: Provides cooking steps for Chicken Fried Rice
```
</details>

## 6. Invoking the Agent (Turn 1)

We supply a `thread_id` in the `config` dictionary. The agent automatically writes its state to that thread's storage.

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="I have some leftover chicken and rice. What can I make?")]},
    config
)

print("Chef Agent Response:")
print("-" * 50)
print(response['messages'][-1].content)

## 7. Inspecting Complete Message History & State Payload

Let's inspect the internal message sequence. Notice how LangChain tracked the `HumanMessage`, the model's intermediate `AIMessage` with `tool_calls`, the corresponding `ToolMessage` with web results, and the final synthesis `AIMessage`.

In [ ]:
from pprint import pprint

print(f"Total messages recorded in thread '1': {len(response['messages'])}")
for i, msg in enumerate(response["messages"]):
    print(f"[{i}] Type: {type(msg).__name__} | Name/ID: {getattr(msg, 'name', 'N/A')}")

print("\nFull state dictionary:")
pprint(response)

## 8. Verifying Thread Memory (Turn 2)

To prove that conversational memory is active, we ask a follow-up question without restating our ingredients. Because `thread_id: "1"` is reused, the agent seamlessly recalls the prior conversation.

In [ ]:
followup_response = agent.invoke(
    {"messages": [HumanMessage(content="Can you give me detailed step-by-step instructions for the first recipe you suggested?")]},
    config
)

print("Follow-up Response (Memory in action):")
print("-" * 50)
print(followup_response['messages'][-1].content)